# 01 — Pillow image editing with model-backed tools

This notebook uses the tool-owned SAM3 and editing pipelines. Models load on first use; install the models extra and configure Hugging Face access before running the agent cells.


In [ ]:
# %pip install -e '.[models]'

In [ ]:
import os
from PIL import Image, ImageDraw
from IPython.display import display
from smolagents import OpenAIModel
from core import Agent
from tools import image_editing_tools

model = OpenAIModel(
    model_id='gemini-2.5-flash',
    api_base='https://generativelanguage.googleapis.com/v1beta/openai/',
    api_key=os.environ['GOOGLE_API_KEY'],
    temperature=0,
)


In [ ]:
image = Image.new('RGB', (512, 320), 'white')
d = ImageDraw.Draw(image)
d.rectangle((60, 80, 180, 220), fill=(220, 60, 60))
d.ellipse((270, 90, 410, 230), fill=(70, 120, 220))
display(image)

box_0 = [55, 75, 185, 225]        # red rectangle
box_1 = [260, 80, 420, 240]       # blue circle region
point_0 = [340, 160]
scribble_0 = [[90, 120], [120, 150], [150, 190]]

In [ ]:
tools = image_editing_tools()

EXAMPLES = '''
Observation:
{"type":"user_message","text":"Remove the region selected by <box>.","objects":["image_0","box_0"]}

Thought: I should create a mask from the selected box, then remove that region.
Act:
mask_0 = segment_object(image_0, box=box_0)
image_1 = remove_object(image_0, mask=mask_0)
send_message(text="Done.", objects=[image_1])
'''

agent = Agent(model=model, tools=tools, examples=EXAMPLES, verbosity_level=1)


In [ ]:
result = agent.send(
        'Use segment_object with <box> to select the region, remove it with remove_object, and send the edited image.',
    objects={'image_0': image, 'box_0': box_0},
)
edited = result.message.objects[0]
display(edited)
print(agent.trajectory_text)


## Continue the same conversation

The generated image variable remains in the Python namespace. The user can refer to the previous result conversationally.

In [ ]:
# Add a new target box on this turn; previous image variables still exist.
result2 = agent.send(
        'Use segment_object with box_1 to select the remaining object, then move it with move_object to target_box_0 and send the result.',
    objects={
        'box_1': box_1,
        'target_box_0': [40, 40, 190, 190],
    },
)
if result2.message.objects:
    display(result2.message.objects[0])
print(agent.trajectory_text)


In [ ]:
steps = agent.raw_steps

for step in steps:
    if "model_input_messages" not in step:
        continue

    print("\n" + "=" * 80)
    print(f"STEP {step.get('step_number', '?')}")
    print("=" * 80)

    for msg in step["model_input_messages"]:
        role = msg.get("role")
        content = msg.get("content")

        print(f"\n[{role}]")

        if isinstance(content, list):
            for block in content:
                if isinstance(block, dict) and block.get("type") == "text":
                    print(block.get("text", ""))
                else:
                    print(block)
        else:
            print(content)
